# OE2-B: reproducción autónoma en Google Colab (latencia, throughput y convergencia por HTTP)

**Objetivo oficial (literal).** Determinar estadísticamente el desempeño propio de la propuesta basada en inteligencia de enjambre, considerando como indicadores el tiempo de convergencia, la latencia y el throughput bajo condiciones experimentales controladas.

**Qué comprueba este cuaderno.** Comprueba las dos mediciones del objetivo en un solo flujo: **componente A** (tiempo de convergencia, latencia y throughput en proceso, 5 000 ciclos) y **componente B** (latencia, throughput y tiempo de convergencia por HTTP, 120 corridas válidas). Para B ejecuta dos etapas en orden: B1, recálculo de los indicadores por corrida desde cada petición registrada; B2, análisis pre-registrado y exploratorio desde los indicadores por corrida. La conclusión final se calcula para ambos componentes.

**Datos de entrada.** Componente A: observaciones por ciclo (`raw_observations.csv`), tiempos de pared por lote, monitor y compuertas de reposo. Componente B: registros por petición, `runs.jsonl` (136 registros finales), `intentos.jsonl`, monitores de los dos segmentos y planes. Scripts de análisis sellados y resultados locales para comparar.

**Qué no hace.** No recalcula los tiempos medidos, que son los del equipo local con interferencia registrada. No repite las campañas ni regenera datos, no modifica archivos originales, scripts ni umbrales, y no convierte ninguna campaña en controlada. Pulse «Ejecutar todo» en un entorno nuevo de Colab: los datos se descargan solos desde GitHub (commit fijado) y se verifican por SHA-256; no hay que subir ni buscar archivos.

**Alcance de este cuaderno.** Solo el componente B del OE2 (por HTTP, 120 corridas válidas). El componente A está en `OE2A_reproduccion_autonoma.ipynb`.

**Contenido.** 1 descripción · 2 entorno y código (clonado de la etiqueta inmutable) · 3 descarga automática y verificación SHA-256 · 4 validación de integridad y estructura · 5 preprocesamiento y cálculo de indicadores (scripts sellados) · 6 análisis estadístico · 7 comparación con el resultado local y con las cifras de la tesis · 8 tablas · 9 gráficos · 10 pruebas automáticas · 11 resumen. Las diferencias se clasifican como exacta, dentro de tolerancia, explicada por versión, no explicada o no reproducible.

## 1. Entorno y código de análisis
El código vive en el repositorio público (módulos legibles en `reproducibility/etapa26_colab_autonomo/src/repro/`); los scripts de análisis sellados vienen dentro de los paquetes de datos, con su hash verificado.

In [ ]:
import os, sys, platform, subprocess, json
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Renato5Lara/multiagent-llm-education.git'
REPO_REF = 'repro-etapa26-v1'            # etiqueta inmutable del codigo; los datos estan fijados por commit en manifests/data_manifest.json
if os.environ.get('REPRO_LOCAL_REPO'):   # solo para pruebas locales antes de publicar la etiqueta
    REPO_DIR = os.environ['REPRO_LOCAL_REPO']
else:
    REPO_DIR = 'repo_repro'
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', '--quiet', '--depth', '1', '--branch', REPO_REF, REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, os.path.join(REPO_DIR, 'reproducibility', 'etapa26_colab_autonomo', 'src'))
import repro
from repro import sh
print('Codigo de analisis:', REPO_URL, '@', REPO_REF)


In [ ]:
try:
    import statsmodels
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'statsmodels'], check=True)
    import statsmodels
import statsmodels, numpy, scipy, pandas, matplotlib
ENTORNO = dict(python=platform.python_version(), plataforma=platform.platform(), numpy=numpy.__version__, scipy=scipy.__version__, pandas=pandas.__version__, matplotlib=matplotlib.__version__, cpu=os.cpu_count(), statsmodels=statsmodels.__version__)
print(ENTORNO)


## 2. Descarga automática y verificación de los paquetes (SHA-256)

In [ ]:
# Descarga automatica desde GitHub (commit fijado) y verificacion SHA-256 contra el manifiesto. No hay que subir nada.
# Si falta un archivo, no se puede descargar o su hash no coincide, se detiene con el archivo, la URL y el hash esperado.
PAQUETES = ['oe2b_colab_paquete', 'oe2b_http_p1', 'oe2b_http_p2']
repro.fetch_packages(PAQUETES, dest='.')


## 3. Verificación interna, cálculo y análisis (celdas del notebook de comprobación publicado, sin cambios salvo la ejecución de scripts)

## Componente B, etapa 1 (B1): indicadores por corrida desde las peticiones registradas

In [ ]:
import hashlib
def sha(p): return hashlib.sha256(open(p,'rb').read()).hexdigest()
bad=[]; n=0
for l in open('http/hashes_paquete_http_oe2b.txt'):
    l=l.strip()
    if not l: continue
    h,nm=l.split(None,1); nm=nm.strip().lstrip('*'); n+=1
    if sha(os.path.join('http',nm))!=h: bad.append(nm)
print('archivos del paquete HTTP verificados contra el hash local:', n, '| distintos:', bad[:5]); assert not bad
import json
ANON={a['archivo']:a['sha256_anonimizado'] for a in json.load(open('http/oe2b/campana_01/anonimizacion_oe2b.json',encoding='utf8'))['archivos']}
base='http/oe2b/campana_01'; ok=0; mal=[]
for l in open(base+'/SHA256SUMS.txt'):
    l=l.strip()
    if not l: continue
    h,nm=l.split(None,1); nm=nm.strip().lstrip('*'); p=os.path.join(base,nm)
    if os.path.exists(p):
        ok+=1
        if sha(p)!=ANON.get(nm,h): mal.append(nm)
print('oe2b: archivos de la campaña presentes y verificados contra su SHA256SUMS.txt (los archivos anonimizados, contra su suma anonimizada):', ok, '| distintos:', mal); assert not mal

In [ ]:
os.makedirs('http/out', exist_ok=True)
for k in ('oe2b',):
    sh(f"python http/recomputo_http.py http/{k}/campana_01 http/out/recomputo_{k}.csv http/out/resumen_{k}.json > http/out/log_{k}.txt 2>&1")
import json
print('Recálculo INDEPENDIENTE, desde cada petición registrada por el generador de carga, de los indicadores por corrida (12 campos) y comparación con lo guardado por el ejecutor:')
for k in ('oe2b',):
    r=json.load(open(f'http/out/resumen_{k}.json'))
    mx=max(r['diferencia_abs_max_por_campo'].values()); mr=max(r['diferencia_rel_max_por_campo'].values())
    print(f"{k}: intentos recalculados {r['intentos_con_raw_recalculados']} | corridas válidas finales recalculadas {r['corridas_validas_finales_recalculadas']} | con los 12 campos idénticos {r['corridas_validas_finales_con_todos_los_campos_iguales']} | diferencia absoluta máxima {mx} | relativa máxima {mr}")

In [ ]:
import pandas as pd, math
def runs_de(k): return {json.loads(l)['label']: json.loads(l) for l in open(f'http/{k}/campana_01/runs.jsonl', encoding='utf8') if l.strip()}
# OE2-B: tabla por carga del informe (media aritmética del P95, throughput y T_conv por corrida; tasa de error acumulada) desde los valores RECALCULADOS
rc=pd.read_csv('http/out/recomputo_oe2b.csv'); rc=rc[rc['valida_final']==True].copy(); rr=runs_de('oe2b')
rc['carga']=rc['label'].map(lambda s: rr[s]['load'])
g=rc.groupby('carga').agg(n=('label','size'), p95=('latency_p95_ms_recalculado','mean'), thr=('throughput_rps_recalculado','mean'), tconv=('t_conv_ms_mean_recalculado','mean'), req=('requests_total_recalculado','sum'), err=('errors_recalculado','sum')).reset_index()
g['error_tasa']=g['err']/g['req']
e=pd.read_csv('http/oe2b/esperado_resumen_por_carga.csv')
m=g.merge(e,on='carga')
d=lambda x,y: float((m[x]-m[y]).abs().max())
print('OE2-B por carga (n=20 corridas válidas): dif abs máx desde HTTP crudo frente a la tabla del análisis | P95 media:', d('p95','p95_ms_media'), '| throughput media:', d('thr','thr_media'), '| T_conv media:', d('tconv','tconv_ms_media'), '| tasa de error:', d('error_tasa_x','error_tasa_y'))
print(m[['carga','n_x','p95','p95_ms_media','thr','thr_media']].to_string(index=False))


## Componente B, etapa 2 (B2): análisis pre-registrado y exploratorio desde los indicadores por corrida

In [ ]:
import hashlib
def sha(p): return hashlib.sha256(open(p,'rb').read()).hexdigest()
bad=[]; n=0
for l in open('oe2b/hashes_paquete_oe2b.txt'):
    l=l.strip()
    if not l: continue
    h,nm=l.split(None,1); nm=nm.strip().lstrip('*'); p=os.path.join('oe2b',nm); n+=1
    if sha(p)!=h: bad.append(nm)
print('archivos del paquete verificados contra el hash local:', n, '| distintos:', bad); assert not bad
import json
ANON={a['archivo']:a['sha256_anonimizado'] for a in json.load(open('oe2b/campana_01/anonimizacion_oe2b.json',encoding='utf8'))['archivos']}
bad2=[]; ok2=[]
for l in open('oe2b/campana_01/SHA256SUMS.txt'):
    l=l.strip()
    if not l: continue
    h,nm=l.split(None,1); nm=nm.strip().lstrip('*'); p=os.path.join('oe2b/campana_01',nm)
    if os.path.exists(p):
        ok2.append(nm)
        if sha(p)!=ANON.get(nm,h): bad2.append(nm)
print('contra el SHA256SUMS.txt de la campana (los seis archivos anonimizados contra su suma anonimizada), archivos presentes:', ok2, '| distintos:', bad2); assert not bad2

## Monitor de recursos de los dos segmentos
Se concatena el monitor del segmento 1 con el del segmento 2 (sin su encabezado), igual que en el análisis local, y se compara con el archivo derivado local.

In [ ]:
seg1=open('oe2b/monitor_segmento1.csv','rb').read()
seg2=open('oe2b/monitor_segmento2.csv','rb').read().split(b'\n',1)[1]
os.makedirs('oe2b/salida_colab',exist_ok=True)
open('oe2b/salida_colab/monitor_concatenado.csv','wb').write(seg1+seg2)
print('monitor concatenado igual al derivado local (SHA-256):', sha('oe2b/salida_colab/monitor_concatenado.csv')==sha('oe2b/esperado_local/monitor_concatenado_derivado.csv'))

## Conteo de corridas (120 válidas, 16 no válidas reemplazadas, intentos)

In [ ]:
import json, pandas as pd
R=[json.loads(l) for l in open('oe2b/campana_01/runs.jsonl',encoding='utf8') if l.strip()]
I=[json.loads(l) for l in open('oe2b/campana_01/intentos.jsonl',encoding='utf8') if l.strip()]
r=pd.DataFrame(R); print('registros finales:', len(r), '| validos:', int((r.valid==True).sum()), '| no validos:', int((r.valid!=True).sum()), '| intentos totales:', len(I))
print(r[r.valid==True].groupby('load').size().to_dict())

## 1. Análisis PRE-REGISTRADO (`analisis_oe2b.py`, mismo script y mismos argumentos que en local)

In [ ]:
sh(f"python oe2b/analisis_oe2b.py oe2b/campana_01/runs.jsonl oe2b/salida_colab/preregistrado oe2b/salida_colab/monitor_concatenado.csv > oe2b/salida_colab/log_preregistrado.txt 2>&1")
print(open('oe2b/salida_colab/log_preregistrado.txt').read()[-700:])

## 2. Análisis EXPLORATORIO (NO pre-registrado; estratificación posterior por Zoom)

In [ ]:
sh(f"python oe2b/analisis_exploratorio_oe2b.py oe2b/campana_01 oe2b/salida_colab/monitor_concatenado.csv oe2b/salida_colab/exploratorio > oe2b/salida_colab/log_exploratorio.txt 2>&1")
print(open('oe2b/salida_colab/log_exploratorio.txt').read()[-900:])

## 3. Comparación Colab vs local (diferencias absolutas y relativas)

In [ ]:
import json, math
import pandas as pd
def hoja(x, p=''):
    if isinstance(x, dict):
        for k,v in x.items(): yield from hoja(v, p+'/'+str(k))
    elif isinstance(x, list):
        for i,v in enumerate(x): yield from hoja(v, p+'/'+str(i))
    else: yield p, x
def comparar_json(ruta_colab, ruta_local, etiqueta):
    A=dict(hoja(json.load(open(ruta_colab,encoding='utf8')))); B=dict(hoja(json.load(open(ruta_local,encoding='utf8'))))
    mismas = set(A)==set(B); mx=0.0; mxrel=0.0; peor=None; n_num=0; n_dist=0; n_nonum=0; dist=[]
    for k in A:
        if k not in B: continue
        va, vb = A[k], B[k]
        if isinstance(va,(int,float)) and isinstance(vb,(int,float)) and not isinstance(va,bool) and not isinstance(vb,bool):
            n_num+=1; dif=abs(va-vb); rel=dif/max(abs(vb),1e-300) if dif>0 else 0.0
            if dif>mx: mx=dif; peor=(k,va,vb)
            mxrel=max(mxrel, rel)
            if dif>0: n_dist+=1
            if dif>1e-9 and rel>1e-9: dist.append((k,va,vb,dif,rel))
        else:
            n_nonum+=1
            if va!=vb: dist.append((k,va,vb,None,None))
    print(f'[{etiqueta}] mismas claves: {mismas} | hojas: {len(A)} (numericas {n_num}, no numericas {n_nonum})')
    print(f'   diferencia absoluta maxima: {mx:.3e} | relativa maxima: {mxrel:.3e} | hojas numericas con diferencia > 0: {n_dist} | hojas con diferencia > 1e-9 (abs y rel): {len(dist)}')
    print('   peor hoja:', peor)
    for d in dist[:20]: print('   DIFERENCIA', d)
    return dict(etiqueta=etiqueta, mismas_claves=mismas, hojas=len(A), numericas=n_num, dif_abs_max=mx, dif_rel_max=mxrel, hojas_con_dif=n_dist, hojas_dif_gt_1e9=len(dist))
def comparar_csv(ruta_colab, ruta_local, etiqueta):
    a=pd.read_csv(ruta_colab); b=pd.read_csv(ruta_local)
    ok = list(a.columns)==list(b.columns) and a.shape==b.shape
    mx=0.0; mxrel=0.0; nd=0
    if ok:
        for c in a.columns:
            if pd.api.types.is_numeric_dtype(a[c]) and pd.api.types.is_numeric_dtype(b[c]) and not pd.api.types.is_bool_dtype(a[c]) and not pd.api.types.is_bool_dtype(b[c]):
                dif=(a[c]-b[c]).abs().fillna(0); mx=max(mx,float(dif.max()))
                rel=(dif/b[c].abs().clip(lower=1e-300)).where(dif>0,0.0); mxrel=max(mxrel,float(rel.max())); nd+=int((dif>0).sum())
            else:
                nd+=int((a[c].astype(object).where(a[c].notna(),'<NA>').astype(str)!=b[c].astype(object).where(b[c].notna(),'<NA>').astype(str)).sum())
    print(f'[{etiqueta}] misma forma y columnas: {ok} {a.shape} | dif abs max: {mx:.3e} | rel max: {mxrel:.3e} | celdas distintas: {nd}')
    return dict(etiqueta=etiqueta, misma_forma=ok, filas=a.shape[0], dif_abs_max=mx, dif_rel_max=mxrel, celdas_distintas=nd)

In [ ]:
res=[]
res.append(comparar_json('oe2b/salida_colab/preregistrado/resultados_oe2b.json','oe2b/esperado_local/resultados_oe2b.json','OE2-B pre-registrado: resultados_oe2b.json'))
res.append(comparar_csv('oe2b/salida_colab/preregistrado/resumen_por_carga.csv','oe2b/esperado_local/resumen_por_carga.csv','OE2-B pre-registrado: resumen_por_carga.csv'))
res.append(comparar_json('oe2b/salida_colab/exploratorio/exploratorio_oe2b.json','oe2b/esperado_local/exploratorio_oe2b.json','OE2-B EXPLORATORIO: exploratorio_oe2b.json'))
res.append(comparar_csv('oe2b/salida_colab/exploratorio/estratos_por_carga.csv','oe2b/esperado_local/estratos_por_carga.csv','OE2-B EXPLORATORIO: estratos_por_carga.csv'))
pd.DataFrame(res).to_csv('oe2b/salida_colab/comparacion_colab_vs_local.csv',index=False)
print(pd.DataFrame(res)[['etiqueta','dif_abs_max','dif_rel_max']].to_string())

## 4. Indicadores principales (Colab | local) y conclusión por requisito

In [ ]:
a=json.load(open('oe2b/salida_colab/preregistrado/resultados_oe2b.json',encoding='utf8')); b=json.load(open('oe2b/esperado_local/resultados_oe2b.json',encoding='utf8'))
print('validas', a['n_corridas_validas'], b['n_corridas_validas'], '| no validas', a['n_no_validas'], b['n_no_validas'], '| intentos extra', a['intentos_extra'], b['intentos_extra'], '| interferidas', a['interferidas'], b['interferidas'], '| generador saturado', a['generador_saturado'], b['generador_saturado'])
rows=[]
for L in ['1','5','10','25','50','100']:
    x=a['resultados']['por_carga'][L]; y=b['resultados']['por_carga'][L]; e=a['resultados']['error'][L]; f=b['resultados']['error'][L]
    rows.append(dict(carga=L, n=x['n_corridas'], p95_colab=round(x['p95_ms']['media'],3), p95_local=round(y['p95_ms']['media'],3), ic_p95_colab=[round(x['p95_ms']['ic95_t_log']['lo'],1),round(x['p95_ms']['ic95_t_log']['hi'],1)], thr_colab=round(x['throughput_rps']['media'],4), thr_local=round(y['throughput_rps']['media'],4), tconv_colab=round(x['t_conv_ms']['media'],3), tconv_local=round(y['t_conv_ms']['media'],3), err_colab=round(e['tasa'],6), err_local=round(f['tasa'],6)))
print(pd.DataFrame(rows).to_string())
for k in ('contrastes_p95','contrastes_throughput'):
    print(k); print(pd.DataFrame({L:{'p_holm_colab':a['resultados'][k][L]['p_holm'],'p_holm_local':b['resultados'][k][L]['p_holm'],'cumple_colab':[v for kk,v in a['resultados'][k][L].items() if kk.startswith('cumple')][0],'cumple_local':[v for kk,v in b['resultados'][k][L].items() if kk.startswith('cumple')][0]} for L in a['resultados'][k]}).T.to_string())
print('deriva por orden (Spearman):', a['resultados']['estabilidad']['deriva_orden_log_p95'], '|', b['resultados']['estabilidad']['deriva_orden_log_p95'])
print('efecto de bloque (Kruskal):', a['resultados']['estabilidad']['efecto_bloque_log_p95_kruskal'], '|', b['resultados']['estabilidad']['efecto_bloque_log_p95_kruskal'])
print('sensibilidad sin interferidas:', a['sensibilidad_sin_interferidas_ni_generador_saturado'])
ea=json.load(open('oe2b/salida_colab/exploratorio/exploratorio_oe2b.json',encoding='utf8')); eb=json.load(open('oe2b/esperado_local/exploratorio_oe2b.json',encoding='utf8'))
print('EXPLORATORIO: con/sin zoom', ea['n_con_zoom'], ea['n_sin_zoom'], '|', eb['n_con_zoom'], eb['n_sin_zoom'])
print('intentos descartados por Locust (peticiones, errores):', [(d['peticiones'],d['errores']) for d in ea['intentos_descartados_locust']], 'local:', [(d['peticiones'],d['errores']) for d in eb['intentos_descartados_locust']])
print('error acumulado:', ea['error_acumulado'], '|', eb['error_acumulado'])

## 4. Comparación con las cifras de la tesis
Cada cifra se lee del resultado **reproducido** y se compara con la que figura en la tesis (versión CIERRE_FINAL) y en el JSON oficial sellado (`manifests/valores_oficiales.json`).

In [ ]:
from repro import official, validation, tables, figures
from repro.data_loading import sha256_file
import pandas as pd
RUTAS = {'resultados_oe2b.json': 'oe2b/salida_colab/preregistrado/resultados_oe2b.json', 'intentos.jsonl': 'oe2b/campana_01/intentos.jsonl'}
cifras = official.evaluar('OE2-B', RUTAS)
print(cifras[['id', 'descripcion', 'esperado', 'obtenido', 'dif', 'estado']].to_string(index=False))


## 5. Clasificación de las diferencias frente al resultado local
Exacta · dentro de tolerancia (1e-9) · explicada por versión · no explicada · no reproducible.

In [ ]:
import scipy
REG = validation.Registro(scipy_version=scipy.__version__)
COMPARACIONES = [('OE2-B resultados_oe2b.json', 'oe2b/salida_colab/preregistrado/resultados_oe2b.json', 'oe2b/esperado_local/resultados_oe2b.json'), ('OE2-B resumen_por_carga.csv', 'oe2b/salida_colab/preregistrado/resumen_por_carga.csv', 'oe2b/esperado_local/resumen_por_carga.csv'), ('OE2-B exploratorio_oe2b.json', 'oe2b/salida_colab/exploratorio/exploratorio_oe2b.json', 'oe2b/esperado_local/exploratorio_oe2b.json'), ('OE2-B estratos_por_carga.csv', 'oe2b/salida_colab/exploratorio/estratos_por_carga.csv', 'oe2b/esperado_local/estratos_por_carga.csv')]
for _e, _a, _b in COMPARACIONES:
    REG.comparar(_e, _a, _b)


## 6. Tablas

In [ ]:
OUT = 'outputs'
TABLAS = {'resumen_por_carga': 'oe2b/salida_colab/preregistrado/resumen_por_carga.csv', 'estratos_por_carga_exploratorio': 'oe2b/salida_colab/exploratorio/estratos_por_carga.csv'}
tablas_generadas = tables.exportar_tablas('OE2-B', TABLAS, os.path.join(OUT, 'tables'))
print('Tablas generadas:')
for _t in tablas_generadas:
    print('  ', _t)


## 7. Gráficos
Se generan desde la salida del análisis ejecutado arriba (no desde valores copiados). Se exportan PNG (300 ppi), SVG y el CSV de la serie dibujada (`figure_data/`).

In [ ]:
METAS = official.cargar_valores()['metas_de_diseno']
fig = figures.fig_oe2(RUTAS['resultados_oe2b.json'], os.path.join(OUT, 'figures', 'OE2-B'), metas=METAS)
ref = figures.fig_oe2('oe2b/esperado_local/resultados_oe2b.json', os.path.join(OUT, 'figures_ref_desde_resultado_local', 'OE2-B'), metas=METAS)
REG.comparar('Serie de la Figura 2 reproducida frente a la serie dibujada desde el resultado oficial local', fig['serie_csv'], ref['serie_csv'])
_t = official.cargar_valores()['figuras_tesis']['fig_oe2']
IDENTICA = sha256_file(fig['png']) == _t['sha256_png_tesis']
print('Figura 2: PNG identico bit a bit al de la tesis (pagina %s del indice): %s' % (_t['pagina_indice'], IDENTICA))
if not IDENTICA:
    print('  Esperable fuera del entorno de la tesis (matplotlib 3.10.3 y fuente Arial): la imagen puede diferir en pixeles; la comprobacion valida es la de la serie dibujada.')
try:
    from IPython.display import Image, display
    display(Image(fig['png']))
except Exception:
    pass


## 8. Pruebas automáticas de consistencia

In [ ]:
fallos = []
if cifras.estado.isin(['DIFIERE', 'NO_EVALUABLE']).any():
    fallos.append('cifras oficiales: ' + str(cifras[cifras.estado.isin(['DIFIERE', 'NO_EVALUABLE'])][['id', 'estado']].values.tolist()))
peor, cuenta = REG.veredicto()
if peor in ('NO_EXPLICADA', 'NO_REPRODUCIBLE'):
    fallos.append('comparacion con el resultado local: ' + peor)
if not (os.path.getsize(fig['png']) > 0 and os.path.getsize(fig['serie_csv']) > 0):
    fallos.append('figura vacia')
assert not fallos, fallos
print('PRUEBAS AUTOMATICAS DE CONSISTENCIA: todas superadas | cifras oficiales comprobadas:', len(cifras), '| comparaciones:', cuenta)


## 9. Resumen final de la reproducción

In [ ]:
import shutil
resumen = tables.guardar_resumen('OE2-B', os.path.join(OUT, 'reports'), REG.tabla(), cifras, ENTORNO, extra=dict(veredicto=peor, conteo=cuenta, paquetes=PAQUETES))
print('RESUMEN DE REPRODUCCION OE2-B')
print(REG.tabla()[['comparacion', 'categoria', 'dif_abs_max']].to_string(index=False))
print('Veredicto (peor categoria):', peor, cuenta)
print('Cifras de la tesis comprobadas: %d (%d coinciden exactamente, %d dentro del redondeo)' % (len(cifras), (cifras.estado == 'COINCIDE').sum(), (cifras.estado == 'COINCIDE_REDONDEO').sum()))
print('LIMITACIONES: los resultados dependen de la carga y de la interferencia (120 de 120 corridas interferidas); no todos los niveles de carga cumplen a la vez las metas; se conservan los 180 intentos (131 correctos, 47 fallos tecnicos, 2 ventanas invalidas).')
print('Alcance: se reproduce el analisis, no la adquisicion de datos; no se repite ninguna campana.')
_z = shutil.make_archive('resultados_OE2-B', 'zip', OUT)
print('Salidas empaquetadas en', os.path.basename(_z))
if IN_COLAB:
    from google.colab import files
    files.download(_z)
